# Tree-based machine-learning models

In [2]:
%pip install xgboost

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

print(f'scikit-learn: {sklearn.__version__}')
print(f'XGBoost: {xgboost.__version__}')

scikit-learn: 1.9.1
XGBoost: 3.4.1


In [4]:
# Loading the processed features
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

features_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_model_features.parquet'
)

comparison_path = (
    project_root
    / 'reports'
    / 'validation_model_comparison.csv'
)

feature_df = pd.read_parquet(
    features_path
)

feature_df['date'] = pd.to_datetime(
    feature_df['date']
)

feature_df = feature_df.sort_values(
    ['store_id', 'item_id', 'date']
).reset_index(drop=True)

print(f'Rows: {len(feature_df):,}')
print(f'Items: {feature_df["item_id"].nunique():,}')
print(
    f'Date range: '
    f'{feature_df["date"].min().date()} '
    f'to {feature_df["date"].max().date()}'
)

Rows: 166,886
Items: 108
Date range: 2011-02-26 to 2016-05-22


In [5]:
# Chronological split
all_dates = pd.DatetimeIndex(
    sorted(feature_df['date'].unique())
)

train_dates = all_dates[:-56]
validation_dates = all_dates[-56:-28]
test_dates = all_dates[-28:]

# Applying train and val splits
training_df = feature_df.loc[
    feature_df['date'].isin(train_dates)
].copy()

validation_df = feature_df.loc[
    feature_df['date'].isin(validation_dates)
].copy()

In [6]:
print(
    f'Training: {train_dates.min().date()} '
    f'to {train_dates.max().date()}'
)

print(
    f'Validation: {validation_dates.min().date()} '
    f'to {validation_dates.max().date()}'
)

print(
    f'Testing: {test_dates.min().date()} '
    f'to {test_dates.max().date()}'
)

print(f'Training rows: {len(training_df):,}')
print(f'Validation rows: {len(validation_df):,}')

assert train_dates.max() < validation_dates.min()
assert validation_dates.max() < test_dates.min()

Training: 2011-02-26 to 2016-03-27
Validation: 2016-03-28 to 2016-04-24
Testing: 2016-04-25 to 2016-05-22
Training rows: 160,838
Validation rows: 3,024


In [7]:
# Define selected predictors for the tree models
numeric_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'sell_price',
    'price_change',
    'event_indicator'
]

categorical_features = [
    'weekday',
    'month',
    'item_id',
    'dept_id',
    'cat_id'
]

model_features = (numeric_features + categorical_features)

target = 'sales'

assert target not in model_features

In [8]:
# Checking for missing values
missing_values = (
    training_df[
        model_features + [target]
    ]
    .isna()
    .sum()
)

missing_values[missing_values > 0]

Series([], dtype: int64)

### Tree model preprocessing

In [9]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')
    )
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder( handle_unknown='ignore'))
])

In [10]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features)
    ], sparse_threshold=1.0
)

### Model

In [11]:
tree_models = {
    'random_forest': Pipeline([
        ('preprocessor',clone(tree_preprocessor)),
        ('regressor', RandomForestRegressor(
                n_estimators=100,
                max_depth=16,
                min_samples_leaf=2,
                max_features='sqrt',
                max_samples=0.8,
                n_jobs=-1,
                random_state=10
            )
        )
    ]),

    'xgboost': Pipeline([
        ('preprocessor', clone(tree_preprocessor)),
        ('regressor', XGBRegressor(
                                n_estimators=300,
                                max_depth=6,
                                learning_rate=0.05,
                                subsample=0.8,
                                colsample_bytree=0.8,
                                objective='reg:squarederror',
                                eval_metric='rmse',
                                tree_method='hist',
                                n_jobs=-1,
                                random_state=10
            )
        )
    ])
}

In [12]:
# Training and timekeeping
X_train = training_df[model_features]

y_train = training_df[target]

trained_tree_models = {}
training_time_rows = []

for model_name, model in tree_models.items():
    print(f'Training {model_name}...')

    start_time = perf_counter()

    model.fit(X_train, y_train)

    elapsed_seconds = (perf_counter() - start_time)

    trained_tree_models[model_name] = model

    training_time_rows.append({
        'model': model_name,
        'training_seconds': elapsed_seconds,
        'training_minutes': (
            elapsed_seconds / 60
        )
    })

    print(
        f'Finished {model_name} in '
        f'{elapsed_seconds / 60:.2f} minutes.'
    )

training_times = pd.DataFrame(training_time_rows)

training_times

Training random_forest...
Finished random_forest in 0.27 minutes.
Training xgboost...
Finished xgboost in 0.04 minutes.


,model,training_seconds,training_minutes
0,random_forest,16.268270,0.271138
1,xgboost,2.338127,0.038969


In [13]:
# Using the same recursive function from the linear model
def recursive_ml_forecast(
    model,
    data,
    forecast_dates,
    feature_columns,
    model_name
):
    forecast_dates = pd.DatetimeIndex(forecast_dates)

    forecast_start = forecast_dates.min()

    histories = {}

    for series_values, group in data.groupby(
        series_keys,
        sort=False
    ):
        history = (
            group.loc[
                group['date'] < forecast_start,
                'sales'
            ]
            .astype(float)
            .tolist()
        )

        if len(history) < 28:
            raise ValueError(
                f'{series_values} has fewer than '
                f'28 historical observations.'
            )

        histories[series_values] = history

    prediction_frames = []

    for forecast_date in forecast_dates:
        day_rows = data.loc[
            data['date'] == forecast_date
        ].sort_values(
            series_keys
        ).copy()

        for row_index, row in day_rows.iterrows():
            series_values = tuple(
                row[column]
                for column in series_keys
            )

            history = histories[series_values]

            day_rows.at[
                row_index, 'lag_1'
            ] = history[-1]

            day_rows.at[
                row_index, 'lag_7'
            ] = history[-7]

            day_rows.at[
                row_index, 'lag_14'
            ] = history[-14]

            day_rows.at[
                row_index, 'lag_28'
            ] = history[-28]

            day_rows.at[
                row_index, 'rolling_mean_7'
            ] = np.mean(history[-7:])

            day_rows.at[
                row_index,
                'rolling_mean_28'
            ] = np.mean(history[-28:])

            day_rows.at[
                row_index,
                'rolling_std_7'
            ] = np.std(
                history[-7:],
                ddof=1
            )

        assert not day_rows[feature_columns].isna().any().any()

        raw_forecasts = model.predict(day_rows[feature_columns])

        # Sales cannot be negative.
        clipped_forecasts = np.maximum(
            raw_forecasts,
            0
        )

        output = day_rows[
            [
                'store_id',
                'item_id',
                'dept_id',
                'cat_id',
                'date',
                'sales'
            ]
        ].copy()

        output['model'] = model_name
        output['raw_forecast'] = raw_forecasts
        output['forecast'] = clipped_forecasts

        prediction_frames.append(output)

        # Use clipped predictions to construct
        # subsequent forecast-day features.
        for row, prediction in zip(
            day_rows.itertuples(index=False),
            clipped_forecasts
        ):
            series_values = (
                row.store_id,
                row.item_id
            )

            histories[
                series_values
            ].append(float(prediction))

    return pd.concat(
        prediction_frames,
        ignore_index=True
    )

In [14]:
# Using prediction metric function from the linear model results
def calculate_prediction_metrics(
    predictions,
    prediction_column
):
    actual = predictions['sales'].to_numpy(dtype=float)

    forecast = predictions[prediction_column].to_numpy(dtype=float)

    errors = actual - forecast

    mae = np.mean(np.abs(errors))

    rmse = np.sqrt(np.mean(errors ** 2))

    denominator = np.sum(np.abs(actual))

    wape = (
        np.sum(np.abs(errors))
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape
    }

In [15]:
series_keys = [
    'store_id',
    'item_id'
]

In [16]:
# Tree-model validation forecasts
tree_prediction_frames = []

for model_name, model in (
    trained_tree_models.items()
):
    print(f'Forecasting with {model_name}...')

    model_predictions = (
        recursive_ml_forecast(
            model=model,
            data=feature_df,
            forecast_dates=validation_dates,
            feature_columns=model_features,
            model_name=model_name
        )
    )

    tree_prediction_frames.append(model_predictions)

validation_tree_predictions = pd.concat(
    tree_prediction_frames,
    ignore_index=True
)

validation_tree_predictions.head()

Forecasting with random_forest...
Forecasting with xgboost...


,store_id,item_id,dept_id,cat_id,date,sales,model,raw_forecast,forecast
0,CA_1,FOODS_1_010,FOODS_1,FOODS,2016-03-28,0,random_forest,0.145686,0.145686
1,CA_1,FOODS_2_016,FOODS_2,FOODS,2016-03-28,2,random_forest,3.672225,3.672225
2,CA_1,FOODS_2_071,FOODS_2,FOODS,2016-03-28,3,random_forest,0.702340,0.702340
3,CA_1,FOODS_2_149,FOODS_2,FOODS,2016-03-28,3,random_forest,1.008607,1.008607
4,CA_1,FOODS_2_246,FOODS_2,FOODS,2016-03-28,0,random_forest,0.097550,0.097550


In [17]:
# Checking the size
expected_prediction_rows = (
    feature_df['item_id'].nunique()
    * 28
    * len(trained_tree_models)
)

print(
    f'Prediction rows: '
    f'{len(validation_tree_predictions):,}'
)

print(
    f'Expected rows: '
    f'{expected_prediction_rows:,}'
)

assert (
    len(validation_tree_predictions)
    == expected_prediction_rows
)

Prediction rows: 6,048
Expected rows: 6,048


In [18]:
# Checking any negative predictions
tree_negative_summary = (
    validation_tree_predictions
    .assign(
        negative_prediction=lambda data:
        data['raw_forecast'] < 0
    )
    .groupby('model')
    .agg(
        total_predictions=(
            'raw_forecast',
            'size'
        ),
        negative_predictions=(
            'negative_prediction',
            'sum'
        ),
        minimum_prediction=(
            'raw_forecast',
            'min'
        )
    )
    .reset_index()
)

tree_negative_summary[
    'negative_percentage'
] = (
    tree_negative_summary[
        'negative_predictions'
    ]
    / tree_negative_summary[
        'total_predictions'
    ]
    * 100
)

tree_negative_summary

,model,total_predictions,negative_predictions,minimum_prediction,negative_percentage
0,random_forest,3024,0,0.073612,0.0
1,xgboost,3024,0,0.048071,0.0


In [19]:
# Validation metrics for the tree models
tree_metric_rows = []

for model_name, group in (
    validation_tree_predictions.groupby(
        'model'
    )
):
    raw_metrics = (
        calculate_prediction_metrics(
            group,
            'raw_forecast'
        )
    )

    clipped_metrics = (
        calculate_prediction_metrics(
            group,
            'forecast'
        )
    )

    tree_metric_rows.append({
        'model': model_name,
        'prediction_version': 'raw',
        **raw_metrics
    })

    tree_metric_rows.append({
        'model': model_name,
        'prediction_version': 'clipped',
        **clipped_metrics
    })

tree_validation_metrics = (
    pd.DataFrame(tree_metric_rows)
    .sort_values(
        ['prediction_version', 'WAPE']
    )
    .reset_index(drop=True)
)

tree_validation_metrics

,model,prediction_version,MAE,RMSE,WAPE
0,random_forest,clipped,1.177787,2.278336,0.751715
1,xgboost,clipped,1.247358,2.408888,0.796119
2,random_forest,raw,1.177787,2.278336,0.751715
3,xgboost,raw,1.247358,2.408888,0.796119


In [28]:
# Comparison with all model stages
previous_comparison = pd.read_csv(comparison_path)

clipped_tree_metrics = (
    tree_validation_metrics.loc[
        tree_validation_metrics[
            'prediction_version'
        ] == 'clipped',
        [
            'model',
            'MAE',
            'RMSE',
            'WAPE'
        ]
    ]
)

In [35]:
previous_columns = [
    'model',
    'MAE',
    'RMSE',
    'WAPE'
]

complete_validation_comparison = (
    pd.concat(
        [
            previous_comparison[
                previous_columns
            ],
            clipped_tree_metrics
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset='model',
        keep='last'
    )
    .sort_values('WAPE')
    .reset_index(drop=True)
)

complete_validation_comparison[
    'WAPE_percentage'
] = (
    complete_validation_comparison[
        'WAPE'
    ]
    * 100
)

complete_validation_comparison

,model,MAE,RMSE,WAPE,WAPE_percentage
0,rolling_mean_28,1.169400,2.417961,0.746363,74.636250
1,random_forest,1.177787,2.278336,0.751715,75.171546
2,xgboost,1.247358,2.408888,0.796119,79.611872
3,linear_regression,1.323802,2.459765,0.844908,84.490836
4,ridge_regression,1.328547,2.462850,0.847937,84.793721
5,seasonal_naive_7,1.432540,3.151173,0.914310,91.430984
6,previous_day,1.507937,3.461905,0.962431,96.243141


In [29]:
# Inspecting the feature importance
importance_frames = []

for model_name, model in (
    trained_tree_models.items()
):
    feature_names = (
        model.named_steps[
            'preprocessor'
        ].get_feature_names_out()
    )

    importances = (
        model.named_steps[
            'regressor'
        ].feature_importances_
    )

    importance_frame = pd.DataFrame({
        'feature': feature_names,
        'importance': importances,
        'model': model_name
    })

    importance_frames.append(importance_frame)

tree_feature_importance = pd.concat(
    importance_frames,
    ignore_index=True
)

In [30]:
def identify_original_feature(transformed_feature):
    numeric_prefix = 'numeric__'
    categorical_prefix = 'categorical__'

    if transformed_feature.startswith(numeric_prefix):
        return transformed_feature.replace(
            numeric_prefix,
            '',
            1
        )

    transformed_feature = (
        transformed_feature.replace(
            categorical_prefix,
            '',
            1
        )
    )

    for feature in categorical_features:
        if transformed_feature.startswith(
            f'{feature}_'
        ):
            return feature

    return transformed_feature

In [31]:
tree_feature_importance[
    'original_feature'
] = (
    tree_feature_importance[
        'feature'
    ].apply(
        identify_original_feature
    )
)

aggregated_importance = (
    tree_feature_importance
    .groupby(
        [
            'model',
            'original_feature'
        ],
        as_index=False
    )['importance']
    .sum()
    .sort_values(
        [
            'model',
            'importance'
        ],
        ascending=[
            True,
            False
        ]
    )
)

aggregated_importance

,model,original_feature,importance
11,random_forest,rolling_mean_7,0.203418
10,random_forest,rolling_mean_28,0.174881
12,random_forest,rolling_std_7,0.105445
7,random_forest,lag_7,0.089071
4,random_forest,lag_1,0.088830
3,random_forest,item_id,0.087746
5,random_forest,lag_14,0.085239
6,random_forest,lag_28,0.062257
14,random_forest,weekday,0.027565
13,random_forest,sell_price,0.025109


In [32]:
# Saving
models_directory = (
    project_root
    / 'models'
)

reports_directory = (
    project_root
    / 'reports'
)

predictions_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_validation_tree_predictions.parquet'
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

reports_directory.mkdir(
    parents=True,
    exist_ok=True
)

predictions_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [36]:
for model_name, model in (
    trained_tree_models.items()
):
    model_path = (
        models_directory
        / f'{model_name}_pipeline.joblib'
    )

    joblib.dump(
        model,
        model_path,
        compress=3
    )

    print(f'Saved model: {model_path}')

validation_tree_predictions.to_parquet(
    predictions_path,
    index=False
)

tree_validation_metrics.to_csv(
    reports_directory
    / 'tree_model_validation_metrics.csv',
    index=False
)

complete_validation_comparison.to_csv(
    reports_directory
    / 'validation_model_comparison_stage8.csv',
    index=False
)

aggregated_importance.to_csv(
    reports_directory
    / 'tree_model_feature_importance.csv',
    index=False
)

training_times.to_csv(
    reports_directory
    / 'tree_model_training_times.csv',
    index=False
)

tree_negative_summary.to_csv(
    reports_directory
    / 'tree_model_negative_predictions.csv',
    index=False
)

print('All Stage 8 models and results saved.')

Saved model: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\random_forest_pipeline.joblib
Saved model: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\xgboost_pipeline.joblib
All Stage 8 models and results saved.
